# DistilBERT implementation

This notebook implements a DistilBERT model for sentiment classification of product reviews. The objective is to fine-tune a pre-trained DistilBERT model and evaluate its performance on the validation set. Different learning rates and numbers of epochs are tested. The best configuration is selected based on validation F1-score. The selected model is then used to generate predictions for the test set.

## Imports

In [1]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from datasets import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

## Load and prepare the data

In [7]:
DATA_DIR = Path("data")

train_df = pd.read_json(DATA_DIR / "train.json")
test_df = pd.read_json(DATA_DIR / "test.json")

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

train_df.head()
train_df.info()
train_df["sentiments"].value_counts()

Training data shape: (7401, 2)
Test data shape: (1851, 1)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7401 entries, 0 to 7400
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reviews     7401 non-null   object
 1   sentiments  7401 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 115.8+ KB


,count
sentiments,
1,6319
0,1082


## Train-validation splits

In [8]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["sentiments"]
)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)

print("Training class distribution:")
print(train_df["sentiments"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(val_df["sentiments"].value_counts(normalize=True))

Training set: (5920, 2)
Validation set: (1481, 2)
Training class distribution:
sentiments
1    0.853885
0    0.146115
Name: proportion, dtype: float64

Validation class distribution:
sentiments
1    0.853477
0    0.146523
Name: proportion, dtype: float64


## Model configuration

In [9]:
MODEL_NAME = "distilbert/distilbert-base-uncased"

bert_configs = [
    {"learning_rate": 2e-5, "epochs": 2},
    {"learning_rate": 1e-5, "epochs": 2},
    {"learning_rate": 2e-5, "epochs": 3},
    {"learning_rate": 1e-5, "epochs": 3}
]

BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Tokenization and dataset preparation

Convert pandas DataFrames into Hugging Face Dataset objects.

In [10]:
train_ds = Dataset.from_pandas(train_df.rename(columns={"sentiments": "label"}), preserve_index=False)
val_ds   = Dataset.from_pandas(val_df.rename(columns={"sentiments": "label"}), preserve_index=False)
test_ds  = Dataset.from_pandas(test_df[["reviews"]], preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

def preprocess_function(examples):
    return tokenizer(examples['reviews'], truncation=True)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

Dataset({
    features: ['reviews', 'label'],
    num_rows: 5920
})
Dataset({
    features: ['reviews', 'label'],
    num_rows: 1481
})
Dataset({
    features: ['reviews'],
    num_rows: 1851
})


Map:   0%|          | 0/5920 [00:00<?, ? examples/s]

Map:   0%|          | 0/1481 [00:00<?, ? examples/s]

Map:   0%|          | 0/1851 [00:00<?, ? examples/s]

In [11]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Evaluation metrics

In [12]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = (precision_recall_fscore_support(
          labels,
          predictions,
          average="binary",
          zero_division=0
          ))

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## Model training and hyperparameter selection

In [13]:
bert_experiment_results = []

best_bert_f1 = -1
best_bert_trainer = None
best_bert_config = None

for i, config in enumerate(bert_configs):

    print(f"\nExperiment {i + 1}/{len(bert_configs)}")
    print("Configuration:", config)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    training_args = TrainingArguments(
        output_dir=f"review_model/experiment_{i + 1}",
        learning_rate=config["learning_rate"],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=config["epochs"],
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        seed=SEED,
        data_seed=SEED,
        push_to_hub=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    results = trainer.evaluate()

    experiment_result = {
        "experiment": i + 1,
        "learning_rate": config["learning_rate"],
        "epochs": config["epochs"],
        "accuracy": results["eval_accuracy"],
        "precision": results["eval_precision"],
        "recall": results["eval_recall"],
        "f1": results["eval_f1"]
    }

    bert_experiment_results.append(experiment_result)

    print("\nValidation results:")
    print(results)

    if results["eval_f1"] > best_bert_f1:
        best_bert_f1 = results["eval_f1"]
        best_bert_trainer = trainer
        best_bert_config = config

print("\nBest configuration:", best_bert_config)
print("Best validation F1:", best_bert_f1)

# Save the best model and tokenizer for later
best_bert_trainer.save_model("review_model/best_distilbert")
tokenizer.save_pretrained("review_model/best_distilbert")
print("Best DistilBERT model saved.")



Experiment 1/4
Configuration: {'learning_rate': 2e-05, 'epochs': 2}


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.178992,0.935854,0.970233,0.954114,0.962106
2,0.187610,0.197026,0.937880,0.963608,0.963608,0.963608


... (25 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.107349,0.196506,3,0.937205,0.955642,0.971519,0.963515



Validation results:
{'eval_loss': 0.19650577008724213, 'eval_accuracy': 0.937204591492235, 'eval_precision': 0.9556420233463035, 'eval_recall': 0.9715189873417721, 'eval_f1': 0.9635151039623382}

Best configuration: {'learning_rate': 2e-05, 'epochs': 3}
Best validation F1: 0.9681978798586572


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best DistilBERT model saved.


## Comparison of DistilBERT configurations

In [14]:
bert_results_df = pd.DataFrame(bert_experiment_results)

bert_results_df = bert_results_df.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

bert_results_df

print("Best DistilBERT configuration:")
print(best_bert_config)

print("\nBest validation F1-score:")
print(best_bert_f1)

Best DistilBERT configuration:
{'learning_rate': 2e-05, 'epochs': 3}

Best validation F1-score:
0.9681978798586572


## Evaluation of the best model

In [15]:
val_output = best_bert_trainer.predict(tokenized_val)

val_predictions = np.argmax(
    val_output.predictions,
    axis=1
)

val_labels = val_output.label_ids

print(classification_report(
        val_labels,
        val_predictions,
        zero_division=0))

              precision    recall  f1-score   support

           0       0.84      0.77      0.80       217
           1       0.96      0.98      0.97      1264

    accuracy                           0.95      1481
   macro avg       0.90      0.87      0.89      1481
weighted avg       0.94      0.95      0.94      1481



In [16]:
cm = confusion_matrix(
    val_labels,
    val_predictions
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[ 167   50]
 [  31 1233]]



## Prediction confidence analysis

In [17]:
## Test set predictions and confidence analysis

test_ds = Dataset.from_pandas(
    test_df[["reviews"]],
    preserve_index=False
)

tokenized_test = test_ds.map(preprocess_function, batched=True)

final_trainer = Trainer(
    model=best_bert_trainer.model,
    processing_class=tokenizer,
    data_collator=data_collator
)

test_logits = final_trainer.predict(tokenized_test).predictions
test_predictions = np.argmax(test_logits, axis=1)

# Generate submission file
submission_df = pd.DataFrame({
    "reviews": test_df["reviews"],
    "sentiments": test_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\nTEST SET PREDICTIONS")
print(f"Test samples: {len(test_predictions)}")
print(f"Predicted positive (1): {(test_predictions == 1).sum()}")
print(f"Predicted negative (0): {(test_predictions == 0).sum()}")

# Calculate prediction confidence
test_confidence = torch.softmax(
    torch.tensor(test_logits),
    dim=1
).max(dim=1).values.numpy()

analysis_df = pd.DataFrame({
    "reviews": test_df["reviews"],
    "prediction": test_predictions,
    "confidence": test_confidence.round(2)
})

pd.set_option("display.max_colwidth", 400)

print("\nLEAST CONFIDENT")
display(analysis_df.sort_values("confidence").head(10))

print("\nMOST CONFIDENT")
display(analysis_df.sort_values("confidence", ascending=False).head(10))

Map:   0%|          | 0/1851 [00:00<?, ? examples/s]


TEST SET PREDICTIONS
Test samples: 1851
Predicted positive (1): 1624
Predicted negative (0): 227

LEAST CONFIDENT


,reviews,prediction,confidence
134,"These shoes were very cute, as I thought they would be from the pictures. But because of the sides being so short, they were nearly impossible to keep on my feet when walking, especially when going up stairs. I returned them after only wearing them for 1 day..",1,0.50
259,I bought the hardtail pants and I do like the way these pants fit. But I really don't like that they fade pretty quickly. I will buy them again. I buy bebe sport and I have to say their pants are pretty good they don't fades as quickly. I notice with their pants they either fit perfectly or they are too long and don't fit as good. I want to try the SO LOW pants,1,0.50
1346,paid double what i found at another site outside of amazon/this store - NEVER AGAIN!,1,0.51
603,"Item ordered was rcvd promptly. I ordered the wrong size, had to return the item and very quickly rcvd the correct size without being charged any additional shipping.",1,0.53
618,This scarf is so soft and beautiful. The picture does not give this scarf justice-the colors are much more vibrant and the material is so ultra soft! I gave this to my aunt for Christmas-she loved it! You cannot believe the quality for this price!,0,0.54
1396,This scarf is so soft and beautiful. The picture does not give this scarf justice-the colors are much more vibrant and the material is so ultra soft! I gave this to my aunt for Christmas-she loved it! You cannot believe the quality for this price!,0,0.54
1008,"i order those shoes because they matched the dress perfectly, then the shoes arrived, they were brown. nothing like the picture on the site",0,0.55
444,May need to get larger size depending on how tight you want it to fi,0,0.55
986,"Champion socks have been my favorite socks for about 10 years now! I refuse to use any other socks. As for the previous review regarding cutting off circulation and leaving marks on your legs, this shouldn't be a problem so long as you don't have massively obese legs. I would suggest the reviewer try losing some weight before knocking on these socks",1,0.56
646,My Male Color Chart does not have Lapis or Heather so I am unsure what color I am ordering. It would be very helpfull to have the colors next to the names in future products,1,0.57



MOST CONFIDENT


,reviews,prediction,confidence
1850,"Most umbrellas handle a light rain well, but leave you with wet pants during heavy rains. This umbrella is essential equipment for keeping you dry during downpours. I use it for the walking part of my commute through Washington, DC thunderstorms and it performs like a champ. It's also big enough to cover two, allowing you to share your dry haven with a child, spouse, or co-worker. The auto...",1,1.0
1849,"These shoers were for my daughter. She loves them! As long as she is happy with them, I am happy with them. Adidas has done a great job with supplying kids shoes for athletics. That is the only kind of shoe, we as a family will wear. Keep up the great work Adidas",1,1.0
1848,These are really cute and very sexy. They will make a nice valentine's day present for the giver and the receiver,1,1.0
2,"everything about the transaction (price, delivery time, quality of item) was great. I wouldn't hesitate to purchase something again from this seller",1,1.0
3,"Not bad for just a shirt. Very durable, and matched my teams colors perfectly. Its just a shirt, but it helped my team and I go on to greatness.....",1,1.0
4,"These are truly wrinkle free and longer than the average womans botton down, which I love!! Overall, these are fabulous shirts and you can't beat the price",1,1.0
5,I love naughty monkey! I'm so happy with their shoes! They don't hurt my feet,1,1.0
1847,"We bought these long-sleeved colored shirts for a child to wear under a rock band tee-shirt. They looked so good that we ordered several more, which he wears on their own as well as under a shirt for the layered look. Fast shipping....top notch seller and product",1,1.0
7,"These shades are a great buy. Fast shipping, great price, and good quality",1,1.0
1846,"These are stylish and nice and perfect to wear under a pair of cutoff Levi's. My cutoffs are so faded, so frayed, so, well you know. So when I peal them off to reveal this Butterfly Lace up Panty (in Black with the pick laces on the back) it's like an instant transformation from outdoor girl to bedroom girl. We only had two colors at our lingerie party, white and the black and pink models, but...",1,1.0
